# Week 1 · Day 1 — Website Summarizer

A first look at calling an LLM from Python. We'll:

1. Set up the OpenAI client and load our API key.
2. Send a simple chat completion ("tell me a joke").
3. Scrape a real web page down to clean text.
4. Ask the model to summarize that page in markdown.

> **Prerequisite:** copy `.env.example` to `.env` and add your `OPENAI_API_KEY`.

---

## 1. Setup & imports

In [1]:
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from scraper import clean_html, fetch_website_content

## 2. Load API credentials

`load_dotenv` reads the key/value pairs from our local `.env` file into
environment variables. We fail fast if the OpenAI key is missing.

In [2]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set")

## 3. Your first chat completion

The Chat Completions API takes a list of `messages`. Each message has a
`role` (`system`, `user`, or `assistant`) and `content`. Here we send a
single user message and print the model's reply.

In [3]:
message = "Hi GPT it's my first time using you, can you tell me a joke?"
messages = [
    {"role": "user", "content": message},
]

In [4]:
# Instantiate the client. It automatically picks up OPENAI_API_KEY from the
# environment, so no key needs to be passed explicitly.
openai = OpenAI()

response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,  # type: ignore
)

# The reply text lives at choices[0].message.content.
display(Markdown(f"**GPT:** {response.choices[0].message.content}"))

**GPT:** Here's a quick one: Why don't scientists trust atoms? Because they make up everything.

Want another? I can tailor to your style—dad jokes, puns, one-liners, or nerdy humor.

## 4. Scrape a website

Before we can summarize a page, we need its content as plain text.
`fetch_website_content` (defined in [scraper.py](scraper.py)) downloads the
page, strips out scripts/styles/images, and returns the title plus visible
text.

In [5]:
my_website_content = fetch_website_content(url="https://azam-sys.netlify.app/")
print(my_website_content)

Title: Azam Mustufa — Software Engineer

Azam Mustufa — Software Engineer
AZAM.DEV
Work
Experience
Connect
mail
System Operational · IBM, Bangalore
Software Engineer
at IBM
Software Engineer at IBM working on SAP SuccessFactors cloud integrations. I build backend systems with Java, Spring Boot, and Node.js, and focus on quality engineering, test automation, and distributed systems.
281+
LeetCode · Top 13%
5
Projects deployed
3×
Hackathon finalist
View Work
arrow_downward
Connect
Projects
Backend Systems
terminal
Live
Database Backup CLI Infrastructure
Production-grade automation CLI for async database backups — reduces manual deployment and migration time by 80%, handles 50GB migrations under 150MB memory with 99.9% success rate.
TypeScript
MongoDB
MySQL
Node.js
Linux
Bash
GitHub
arrow_outward
hub
Live
Project Management Engine
Scalable REST backend with 25+ endpoints and granular RBAC. API responses optimised to under 120ms via Redis caching. BullMQ async pipelines cut cache latency b

## 5. Summarize the page with an LLM

This is a classic **system + user prompt** pattern:

- The **system prompt** sets the model's role and output format.
- The **user prompt** carries the actual data (the scraped page text).

In [6]:
# Defines the assistant's role and the format we want back. No interpolation
# needed here, so it's a plain string (not an f-string).
system_prompt = (
    "You are a helpful assistant that summarizes website content. "
    "Provide a concise summary of the main points and topics covered on the "
    "website. Respond in markdown format."
)

In [7]:
# The user prompt carries the scraped page text for the model to summarize.
user_prompt = f"Here is the content of the website: {my_website_content}"

In [8]:
from typing import Dict, List


def summarize_website_content(messages: List[Dict[str, str]]) -> str | None:
    """Send a prepared messages list to the model and return the reply text.

    Args:
        messages: Chat messages (system + user) describing the task and data.

    Returns:
        The model's response content, or ``None`` if the model returned no text.
    """
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=messages,  # type: ignore
    )
    return response.choices[0].message.content

In [9]:
# Combine both prompts into the messages list the API expects.
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [10]:
# Send the system + user prompts together and render the markdown summary.
response = summarize_website_content(messages)

display(Markdown(data=f"**GPT:** {response}"))

**GPT:** ## Azam Mustufa — Software Engineer

### Summary
- Backend-focused software engineer with strong experience in Java, Spring Boot, and Node.js. Worked on SAP SuccessFactors cloud integrations at IBM. Emphasizes quality engineering, test automation, and distributed systems.

### Experience
- IBM, Bangalore — Software Engineer (Dec 2025 — Present)
  - Building backend services for SAP SuccessFactors cloud integrations using Java, Spring Data JPA, Hibernate, Spring Security + JWT.
  - Implemented asynchronous processing (@Async, @RabbitListener) and data consistency with @Transactional.
  - Wrote unit/integration tests (JUnit, Mockito); resolved 30+ defects, improving release stability by ~15%.

### Projects (selected highlights)
- Production-grade Database Backup CLI Infrastructure
  - Automation CLI for async database backups; 50GB migrations under 150MB memory; 99.9% success; reduces deployment/migration time by 80%.
  - Tech: TypeScript, MongoDB, MySQL, Node.js, Linux, Bash.

- Project Management Engine
  - Scalable REST backend with 25+ endpoints; granular RBAC; API responses under 120ms via Redis; BullMQ pipelines boosting throughput.
  - Tech: Express, MongoDB, Redis, Docker, BullMQ, JWT.

- Distributed Image Processing Backend
  - Scalable image processing/delivery with dynamic transformations; reduces payloads by 40%; faster queries via Mongoose; streaming uploads under load.
  - Tech: Bun, Node.js, Express, MongoDB, Mongoose, Sharp.

- HTTP Caching Proxy
  - CLI-driven proxy caching GET responses to disk; non-GETs forwarded; includes X-Cache header.
  - Tech: TypeScript, Node.js, Express.

- UPI Offline Mesh
  - Spring Boot backend simulating UPI payments over Bluetooth without internet; hybrid RSA-OAEP + AES-GCM encryption; atomic idempotency with ConcurrentHashMap as Redis SETNX.
  - Tech: Java, Spring Boot, RSA/AES-GCM, H2, JUnit.

### Skills
- Languages: Java, JavaScript, TypeScript, Python, C++, SQL
- Frameworks/Libraries: Spring Boot, Spring Security, Node.js, Express, Bun, BullMQ, Mongoose
- Databases/DevOps/Testing: MongoDB, MySQL, Redis, RabbitMQ, Docker, Linux, CI/CD, Git
- Testing: JUnit, Mockito, Selenium, Playwright, Cypress

### Education
- B.E. Computer Science & Engineering, Visvesvaraya Technological University, Belgavi, India
  - CGPA: 7.28/10
  - Core courses: Operating Systems, Computer Networks, OOP, DBMS, Data Structures & Algorithms, Software Engineering
  - Hackathons: 3× Finalist (1 regional, 2 university-level)

### Extra
- LeetCode: 281+ problems, Top 13%
- Hackathon: 3× Finalist
- Location: Bangalore, India; Open to remote
- Contact: azammustafa66@gmail.com | GitHub | LinkedIn | LeetCode | Resume (View/Download)

If you’d like, I can tailor this into a shorter bio for a resume or LinkedIn summary.

## 6. Reuse the helper on another page

Now that `summarize_website_content` is a function, summarizing a different
site is just: scrape → build messages → call the helper.

In [11]:
content = fetch_website_content(url="https://cnn.com/")

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Here is the content of the website: {content}"},
]

In [12]:
summary = summarize_website_content(messages)
display(Markdown(data=f"**GPT:** {summary}"))

**GPT:** - Overview: CNN’s homepage centers on breaking news, live coverage, and a wide range of topics, with a strong emphasis on video content and real-time updates.

- Main navigation topics:
  - US, World, Politics, Business, Health, Entertainment, Style, Travel, Sports, Science, Climate, Weather
  - World Cup 2026, Ukraine-Russia War, Israel-Hamas War
  - Games (puzzle and quick games)

- Content types you’ll find:
  - Breaking News and Live Updates
  - Live TV and Listen (audio/video) options
  - CNN Exclusives and Investigations
  - Analysis and Features
  - Videos, Photos, and multimedia stories
  - CNN 10, Podcasts, and CNN Shorts
  - Special sections like “Inside Africa,” “CNN Heroes,” and other CNN Originals

- Personalization and account features:
  - Sign in / My Account, Settings
  - Newsletters and Topics you follow
  - Download the CNN app (iOS/Android) and access content on the go

- Edition and regional options:
  - Editions for US, International, Arabic, Español
  - Regional/category menus (World, Africa, Americas, Asia, Europe, Middle East, etc.)

- Additional site elements:
  - Ad feedback prompts and feedback forms
  - “Catch up on today’s global news” and “Top stories” carousels
  - Photo and video galleries, as well as live updates from ongoing events

- Example content themes you might encounter:
  - Major U.S. political and economic news
  - Ongoing conflicts and global security updates
  - Health, science, and space developments
  - Entertainment, fashion, and culture coverage
  - Sports highlights, including major events like the World Cup
  - Climate and environmental reporting

- Footer and policy links (typical):
  - Terms of Use, Privacy Policy, Ad Choices, Accessibility & CC, About, Newsletters, Transcripts

This summary captures the core structure and topics of CNN’s homepage as shown in the provided content.

## 7. Selenium-based web scraping

The `requests`-based scraper only sees the initial HTML. For sites that
render their content with JavaScript (single-page apps, infinite scroll,
etc.), we need a real browser. **Selenium** drives a real Chrome instance so
we capture the fully-rendered page, then we reuse `clean_html` to keep the
token count down.

The `WebsiteSummarizer` class below bundles the whole flow — fetch, summarize,
display — behind a small object.

> **Requires:** `selenium` (already in `requirements.txt`). Selenium 4.6+ auto-downloads
> a matching chromedriver, so no manual driver setup is needed — just a local Chrome install.
> A browser window will open while it loads; add `options.add_argument("--headless=new")`
> to run without one.

In [13]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


class WebsiteSummarizer:
    """Fetch a JS-rendered page with Selenium and summarize it with an LLM.

    Typical usage:
        ws = WebsiteSummarizer(url)
        ws.selenium_fetch_content()
        ws.summarize_content()
        ws.display_summary()
    """

    def __init__(self, url: str, base_url: str = "https://api.openai.com/v1", api_key: str | None = None, model: str = "gpt-5-nano") -> None:
        self.url = url
        self.html: str | None = None 
        self.summary: str | None = None 
        self.client = OpenAI(base_url=base_url, api_key=api_key)
        self.model = model

    def selenium_fetch_content(self) -> None:
        """Load the page in headless Chrome and capture the cleaned text."""
        options = Options()
        options.add_argument(
            "User-Agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        )
        driver = webdriver.Chrome(options=options)
        try:
            driver.get(url=self.url)
            driver.implicitly_wait(5)
            # Strip the rendered HTML down to title + visible text so we send far fewer tokens to the model.
            self.html = clean_html(html=driver.page_source)
        except Exception as e:
            print(f"Error fetching content with Selenium: {e}")
        finally:
            driver.quit()

    def summarize_content(self) -> None:
        """Summarize the fetched content using the LLM."""
        if not self.html:
            raise ValueError(
                "Content not fetched yet. Call selenium_fetch_content() first."
            )

        system_prompt = (
            "You are a helpful assistant that summarizes website content. "
            "Provide a concise summary of the main points and topics covered on the "
            "website. Respond in markdown format."
        )
        user_prompt = f"Here is the content of the website: {self.html}"

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,  # type: ignore
        )
        self.summary = response.choices[0].message.content

    def display_summary(self) -> None:
        """Render the summary as markdown."""
        if not self.summary:
            raise ValueError(
                "Summary not generated yet. Call summarize_content() first."
            )
        display(Markdown(data=self.summary))

In [14]:
ws = WebsiteSummarizer(url="https://www.netflix.com/")
ws.selenium_fetch_content()
ws.summarize_content()
ws.display_summary()

- What it is: Netflix India is a streaming service offering unlimited movies, shows, anime, documentaries, and Netflix originals. It’s ad-free and billed monthly with flexible cancellations.
- Pricing: Plans range from ₹149 to ₹649 per month.
- How to watch: Available on web and a wide range of devices (smart TVs, Chromecast, PlayStation, Xbox, Apple TV, mobile devices, etc.). Sign in or create a membership by entering your email.
- Offline viewing: Download shows/movies to watch offline.
- Multi-device and profiles: Stream on multiple devices; create profiles, including kid-friendly profiles with PIN-protected parental controls.
- Content library: Extensive library including feature films, documentaries, Netflix originals, anime, and more; new content added weekly.
- Kids experience: A dedicated kids space with parental controls to manage what kids watch.
- Getting started and support: Help Center, account management, and contact options (including a provided phone number).
- FAQs covered: What Netflix is, pricing, where to watch, how to cancel, what you can watch, and kid-friendly features.
- Access and localization: Language options include English and Hindi; page includes standard Netflix terms, privacy, and legal notices.
- Privacy and cookies: Detailed cookie preferences and privacy information; reCAPTCHA security on the page; notes on cookie types (essential, performance, third-party, advertising) and how to manage them.
- Extra resources: Links to “Ways to Watch,” corporate information, investor relations, jobs, and speed test.

## 8. Point the summarizer at any OpenAI-compatible provider

`WebsiteSummarizer` now takes `base_url`, `api_key`, and `model`, so the exact
same scrape → summarize → display flow can run against a **local model via
Ollama** (or any OpenAI-compatible endpoint) instead of OpenAI — no code
changes, just different constructor arguments.

In [ ]:
ws = WebsiteSummarizer(url="https://dhan.co", base_url="http://localhost:11434/v1", api_key="ollama", model="llama3.2:3b")
ws.selenium_fetch_content()
ws.summarize_content()
ws.display_summary()

Dhan is an online trading platform that provides a wide range of services to retail traders and investors in India. It offers six products within its ecosystem:

1. Dhan App: A mobile app for buying and selling equity, futures, options, commodity, and currency.
2. Dhan Web: A web trading platform for trading across various segments.
3. Options Trader App: A dedicated option trading app for F&O traders.
4. Options Trader Web: A web-based option trading platform for F&O traders.
5. Dhan + TradingView: Integration with TradingView charts, allowing users to trade in Stocks, Futures, and Commodities directly from the TradingView platform.
6. Deposit Participant (DP): Dhan is registered as a Depository Participant (DP) under SEBI and provides depository services for clients.

Dhan aims to provide an innovative experience for retail traders and investors in India, with features such as:

* User-first approach
* Modern trading platform
* Access to TradingView charts
* Options trading app
* Risk management tools

The platform is registered with regulatory bodies such as SEBI, BSE, NSE, MCX, and CDSL, and provides a range of services including brokerage services, research reports, and customer support.

Some key benefits of using Dhan include:

* Easy access to trading platforms and research tools
* Convenience of transacting in multiple languages
* Ability to track and manage investments through the Dhan app
* Access to online investment options
* Competitive brokerage rates

However, as with any investment platform, there are risks involved. Investors should be aware of the following:

* Market risks: Stock market investments carry inherent risks.
* Regulatory risks: Changes in regulatory policies can impact trading activities.
* Customer support risks: Poor customer support can affect user experience.

To mitigate these risks, Dhan provides various tools and resources, such as education modules, risk management strategies, and customer support services.

**Disclaimer:** All communications with the client via chat, phone, or email are for support purposes only. Any commitments or statements made by the agent (human or virtual) shall not be binding on the company.